# Phenotypic Selection Tutorial - Hybrid Breeding

This notebook replicates the AlphaSimR hybrid breeding phenotypic selection tutorial using AlphaSimPy.
It demonstrates phenotypic selection in a hybrid maize breeding program with separate male and female heterotic pools.

**Authors**: Translated from AlphaSimR tutorial by Jon Bancic, Philip Greenspoon, Chris Gaynor, Gregor Gorjanc  
**Date**: 2024  
**Package**: AlphaSimPy

## Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from AlphaSimPy import (
    run_macs, SimParam, new_pop, rand_cross, set_pheno, select_ind,
    mean_g, var_g, make_dh, hybrid_cross, set_pheno_gca, calc_gca, merge_pops
)

print("AlphaSimPy Hybrid Breeding - Phenotypic Selection Tutorial")
print("All libraries imported successfully!")

## Global Parameters

Set up the simulation parameters for the hybrid maize breeding program.

In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1  # Number of simulation replicates
n_burnin = 20  # Number of years in burnin phase
n_future = 20  # Number of years in future phase
n_cycles = n_burnin + n_future
start_tp = 18  # Year to start training population

# Genome simulation
n_chr = 15  # Number of chromosomes
n_qtl = 300  # Number of QTL per chromosome
n_snp = 400  # Number of SNP per chromosome
n_gen_split = 100  # Heterotic pool split

# Initial inbred parents mean and variance
init_mean_g = 70  # bushels per acre
init_var_g = 20  # bushels per acre
# Degree of dominance
mean_dd = 0.92  # mean
var_dd = 0.3  # variance
# Error variances
init_var_ge = 40  # Genotype-by-year interaction
var_e = 270  # Yield trial error variance, bushels per acre
            # Relates to error variance for an entry mean

# Breeding program details
n_parents = 50  # Number of parents to start a breeding cycle
n_crosses = 80  # Number of crosses per year
fam_max = 15  # The maximum number of DH lines per cross
n_dh = 50  # DH lines produced per cross

# Effective replication of yield trials
rep_yt1 = 1  # h2 = 0.06
rep_yt2 = 2  # h2 = 0.11
rep_yt3 = 4  # h2 = 0.20
rep_yt4 = 8  # h2 = 0.34
rep_yt5 = 100  # h2 = 0.86

# Selection on GCA
# Number of inbreds per heterotic pool per stage
n_inbred1 = n_crosses * n_dh  # Do not change
n_inbred2 = 400
n_inbred3 = 40

# Number of testers per heterotic pool per stage
# Values must be smaller than n_elite
n_tester1 = 1
n_tester2 = 3

# Yield trial entries
n_yt1 = n_inbred1 * n_tester1  # Do not change
n_yt2 = n_inbred2 * n_tester2  # Do not change

# Selection on SCA

# Elite parents per heterotic pool
n_elite = 5

# Elite YT size
n_yt3 = n_inbred3 * n_elite  # Do not change
n_yt4 = 20
n_yt5 = 4

scenario_name = "HybridPheno"

print(f"Simulation Parameters:")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Parents per pool: {n_parents}")
print(f"  Crosses per year: {n_crosses}")
print(f"  DH lines per cross: {n_dh}")

## Create Founders

Generate the initial founder population with haplotypes and set up simulation parameters.
The founders are split into two heterotic pools (male and female) to simulate hybrid breeding.

In [ ]:
print("Creating founders...")

# Create founder population
# Split parameter creates two heterotic pools separated by n_gen_split generations
founder_pop = run_macs(
    n_ind=n_parents * 2,
    n_chr=n_chr,
    seg_sites=n_qtl + n_snp,
    inbred=True,
    split=n_gen_split,
    species="MAIZE"
)

print(f"✓ Created founder population: {founder_pop.n_ind} individuals")

# Set simulation parameters
SP = SimParam(founder_pop)

# Restrict segregating sites (separate QTL and SNP)
SP.restrSegSites(minQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)
    print(f"✓ Added SNP chip: {SP.n_snp_chips} SNP chips")

# Add traits: trait represents yield
# Using addTraitADG for additive, dominance, and GxE effects
SP.addTraitADG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    meanDD=mean_dd,
    varDD=var_dd,
    var_gxE=init_var_ge
)
print(f"✓ Added TraitADG: {SP.n_traits} traits")

# Set permanent yield trial error variance
SP.setVarE(var_e=var_e)
print(f"✓ Set error variance: {var_e}")

# Split heterotic pools to form initial parents
FemaleParents = new_pop(founder_pop[0:n_parents], sim_param=SP)
MaleParents = new_pop(founder_pop[n_parents:(n_parents * 2)], sim_param=SP)

print(f"✓ Created female parents: {FemaleParents.n_ind} individuals")
print(f"✓ Created male parents: {MaleParents.n_ind} individuals")

# Set hybrid parents for later yield trials
MaleElite = select_ind(MaleParents, n_ind=n_elite, use="gv", sim_param=SP)
FemaleElite = select_ind(FemaleParents, n_ind=n_elite, use="gv", sim_param=SP)

# Reverse order to keep best parent in longer
MaleElite = select_ind(MaleElite, n_ind=n_elite, use="gv", sim_param=SP)
FemaleElite = select_ind(FemaleElite, n_ind=n_elite, use="gv", sim_param=SP)
# Reverse the order by selecting in reverse
MaleElite_ids = MaleElite.id[::-1]
FemaleElite_ids = FemaleElite.id[::-1]
MaleElite = select_ind(MaleElite, n_ind=n_elite, parents=[MaleElite.id.index(id) for id in MaleElite_ids], sim_param=SP)
FemaleElite = select_ind(FemaleElite, n_ind=n_elite, parents=[FemaleElite.id.index(id) for id in FemaleElite_ids], sim_param=SP)

# Set initial testers for YT1 and YT2
# Requires nTesters to be smaller than nElite
MaleTester1 = select_ind(MaleElite, n_ind=n_tester1, parents=list(range(n_tester1)), sim_param=SP)
FemaleTester1 = select_ind(FemaleElite, n_ind=n_tester1, parents=list(range(n_tester1)), sim_param=SP)
MaleTester2 = select_ind(MaleElite, n_ind=n_tester2, parents=list(range(n_tester2)), sim_param=SP)
FemaleTester2 = select_ind(FemaleElite, n_ind=n_tester2, parents=list(range(n_tester2)), sim_param=SP)

print(f"✓ Created elite parents: {n_elite} per pool")
print(f"✓ Created testers: {n_tester1} for YT1, {n_tester2} for YT2")

print(f"\nFounder population summary:")
print(f"  Mean genetic value (female): {mean_g(FemaleParents)[0]:.3f}")
print(f"  Mean genetic value (male): {mean_g(MaleParents)[0]:.3f}")
print(f"  Genetic variance (female): {var_g(FemaleParents)[0]:.3f}")
print(f"  Genetic variance (male): {var_g(MaleParents)[0]:.3f}")

## Fill Breeding Pipeline

Set up the initial breeding pipeline with 6 stages representing different evaluation years.
The pipeline includes:
- Stage 1: F1 crosses
- Stage 2: Doubled haploid (DH) lines and YT1 (GCA evaluation)
- Stage 3: YT2 (GCA evaluation)
- Stage 4: YT3 (Hybrid evaluation)
- Stage 5: YT4 (Hybrid evaluation)
- Stage 6: YT5 (Hybrid evaluation)

**Note**: Year effects (p parameter) are not yet fully supported in AlphaSimPy's `setPheno` function.
The GxE variance is still included in the trait definition, which affects genetic values.

In [ ]:
print("Filling breeding pipeline...")

# Set initial yield trials with unique individuals
# Sample year effects
P = np.random.uniform(size=6)

# Breeding program
for cohort in range(1, 7):
    print(f"  FillPipeline year: {cohort} of 6")
    
    # Stage 1
    MaleF1 = rand_cross(MaleParents, n_crosses=n_crosses, sim_param=SP)
    FemaleF1 = rand_cross(FemaleParents, n_crosses=n_crosses, sim_param=SP)
    
    # Stage 2
    if cohort < 6:
        p = P[6 - cohort]
        
        MaleDH = make_dh(MaleF1, n_dh=n_dh, sim_param=SP)
        FemaleDH = make_dh(FemaleF1, n_dh=n_dh, sim_param=SP)
        
        MaleYT1 = set_pheno_gca(MaleDH, FemaleTester1, reps=rep_yt1, inbred=True, sim_param=SP)
        FemaleYT1 = set_pheno_gca(FemaleDH, MaleTester1, reps=rep_yt1, inbred=True, sim_param=SP)
    
    # Stage 3
    if cohort < 5:
        p = P[5 - cohort]
        
        MaleYT2 = select_ind(MaleYT1, n_ind=n_inbred2, use="pheno", sim_param=SP)
        FemaleYT2 = select_ind(FemaleYT1, n_ind=n_inbred2, use="pheno", sim_param=SP)
        
        MaleYT2 = set_pheno_gca(MaleYT2, FemaleTester2, reps=rep_yt2, inbred=True, sim_param=SP)
        FemaleYT2 = set_pheno_gca(FemaleYT2, MaleTester2, reps=rep_yt2, inbred=True, sim_param=SP)
    
    # Stage 4
    if cohort < 4:
        p = P[4 - cohort]
        
        MaleInbredYT3 = select_ind(MaleYT2, n_ind=n_inbred3, use="pheno", sim_param=SP)
        FemaleInbredYT3 = select_ind(FemaleYT2, n_ind=n_inbred3, use="pheno", sim_param=SP)
        
        MaleHybridYT3 = hybrid_cross(MaleInbredYT3, FemaleElite, return_hybrid_pop=True, sim_param=SP)
        FemaleHybridYT3 = hybrid_cross(FemaleInbredYT3, MaleElite, return_hybrid_pop=True, sim_param=SP)
        
        MaleHybridYT3 = set_pheno(MaleHybridYT3, var_e=var_e, reps=rep_yt3, sim_param=SP)
        FemaleHybridYT3 = set_pheno(FemaleHybridYT3, var_e=var_e, reps=rep_yt3, sim_param=SP)
    
    # Stage 5
    if cohort < 3:
        p = P[3 - cohort]
        
        MaleHybridYT4 = select_ind(MaleHybridYT3, n_ind=n_yt4, use="pheno", sim_param=SP)
        FemaleHybridYT4 = select_ind(FemaleHybridYT3, n_ind=n_yt4, use="pheno", sim_param=SP)
        
        MaleHybridYT4 = set_pheno(MaleHybridYT4, var_e=var_e, reps=rep_yt4, sim_param=SP)
        FemaleHybridYT4 = set_pheno(FemaleHybridYT4, var_e=var_e, reps=rep_yt4, sim_param=SP)
        
        # Extract inbred parents from hybrid YT4
        MaleInbredYT4_ids = list(set(MaleHybridYT4.mother))
        FemaleInbredYT4_ids = list(set(FemaleHybridYT4.mother))
        MaleInbredYT4 = select_ind(MaleInbredYT3, n_ind=len(MaleInbredYT4_ids), 
                                 parents=[MaleInbredYT3.id.index(id) for id in MaleInbredYT4_ids if id in MaleInbredYT3.id], sim_param=SP)
        FemaleInbredYT4 = select_ind(FemaleInbredYT3, n_ind=len(FemaleInbredYT4_ids),
                                   parents=[FemaleInbredYT3.id.index(id) for id in FemaleInbredYT4_ids if id in FemaleInbredYT3.id], sim_param=SP)
    
    # Stage 6
    if cohort < 2:
        p = P[2 - cohort]
        
        MaleHybridYT5 = select_ind(MaleHybridYT4, n_ind=n_yt5, use="pheno", sim_param=SP)
        FemaleHybridYT5 = select_ind(FemaleHybridYT4, n_ind=n_yt5, use="pheno", sim_param=SP)
        
        MaleHybridYT5 = set_pheno(MaleHybridYT5, var_e=var_e, reps=rep_yt5, sim_param=SP)
        FemaleHybridYT5 = set_pheno(FemaleHybridYT5, var_e=var_e, reps=rep_yt5, sim_param=SP)
        
        # Extract inbred parents from hybrid YT5
        MaleInbredYT5_ids = list(set(MaleHybridYT5.mother))
        FemaleInbredYT5_ids = list(set(FemaleHybridYT5.mother))
        MaleInbredYT5 = select_ind(MaleInbredYT4, n_ind=len(MaleInbredYT5_ids),
                                parents=[MaleInbredYT4.id.index(id) for id in MaleInbredYT5_ids if id in MaleInbredYT4.id], sim_param=SP)
        FemaleInbredYT5 = select_ind(FemaleInbredYT4, n_ind=len(FemaleInbredYT5_ids),
                                   parents=[FemaleInbredYT4.id.index(id) for id in FemaleInbredYT5_ids if id in FemaleInbredYT4.id], sim_param=SP)

print("\nPipeline filled successfully!")

## Main Simulation Loop

Run the breeding program simulation with burn-in and future phases.

In [ ]:
# Create list to store results from reps
results = []

for REP in range(1, n_reps + 1):
    print(f"Working on REP: {REP}")
    
    # Create a data frame to track key parameters
    output = {
        'year': list(range(1, n_cycles + 1)),
        'rep': [REP] * n_cycles,
        'scenario': [scenario_name] * n_cycles,
        'mean_g_inbred': [0.0] * n_cycles,
        'var_g_inbred': [0.0] * n_cycles,
        'mean_g_hybrid': [0.0] * n_cycles,
        'var_g_hybrid': [0.0] * n_cycles,
        'acc_sel': [0.0] * n_cycles,
        'cor': [0.0] * n_cycles
    }
    
    # Simulate year effects
    P = np.random.uniform(size=n_cycles)
    
    # Burn-in phase
    print("--> Working on Burn-in")
    for year in range(1, n_burnin + 1):
        print(f" Working on burnin year: {year}")
        
        # Update parents (pick new parents)
        # Replace 10 oldest inbred parents with 10 new inbreds from YT4 stage
        if year > 1:
            # Keep oldest 40, add 10 new from YT4
            MaleParents_new = select_ind(MaleInbredYT4, n_ind=10, use="pheno", sim_param=SP)
            MaleParents_old = select_ind(MaleParents, n_ind=n_parents - 10, 
                                      parents=list(range(10, n_parents)), sim_param=SP)
            MaleParents = merge_pops([MaleParents_old, MaleParents_new])
            
            FemaleParents_new = select_ind(FemaleInbredYT4, n_ind=10, use="pheno", sim_param=SP)
            FemaleParents_old = select_ind(FemaleParents, n_ind=n_parents - 10,
                                        parents=list(range(10, n_parents)), sim_param=SP)
            FemaleParents = merge_pops([FemaleParents_old, FemaleParents_new])
        
        # Update testers (pick new testers)
        # Replace oldest hybrid parent with parent of best hybrid from YT5
        if year > 1:
            # Find best male inbred from YT5
            best_male_idx = np.argmax(MaleHybridYT5.pheno[:, 0])
            best_male_inbred_id = MaleHybridYT5.mother[best_male_idx]
            best_male_inbred_idx = MaleInbredYT5.id.index(best_male_inbred_id)
            MaleElite_new = select_ind(MaleInbredYT5, n_ind=1, parents=[best_male_inbred_idx], sim_param=SP)
            MaleElite_old = select_ind(MaleElite, n_ind=n_elite - 1, parents=list(range(1, n_elite)), sim_param=SP)
            MaleElite = merge_pops([MaleElite_old, MaleElite_new])
            
            # Find best female inbred from YT5
            best_female_idx = np.argmax(FemaleHybridYT5.pheno[:, 0])
            best_female_inbred_id = FemaleHybridYT5.mother[best_female_idx]
            best_female_inbred_idx = FemaleInbredYT5.id.index(best_female_inbred_id)
            FemaleElite_new = select_ind(FemaleInbredYT5, n_ind=1, parents=[best_female_inbred_idx], sim_param=SP)
            FemaleElite_old = select_ind(FemaleElite, n_ind=n_elite - 1, parents=list(range(1, n_elite)), sim_param=SP)
            FemaleElite = merge_pops([FemaleElite_old, FemaleElite_new])
            
            # Update testers
            MaleTester1 = select_ind(MaleElite, n_ind=n_tester1, parents=list(range(n_tester1)), sim_param=SP)
            FemaleTester1 = select_ind(FemaleElite, n_ind=n_tester1, parents=list(range(n_tester1)), sim_param=SP)
            MaleTester2 = select_ind(MaleElite, n_ind=n_tester2, parents=list(range(n_tester2)), sim_param=SP)
            FemaleTester2 = select_ind(FemaleElite, n_ind=n_tester2, parents=list(range(n_tester2)), sim_param=SP)
        
        # Advance year (advances yield trials by a year)
        p = P[year - 1]
        
        # Stage 6
        MaleHybridYT5 = select_ind(MaleHybridYT4, n_ind=n_yt5, use="pheno", sim_param=SP)
        FemaleHybridYT5 = select_ind(FemaleHybridYT4, n_ind=n_yt5, use="pheno", sim_param=SP)
        
        MaleHybridYT5 = set_pheno(MaleHybridYT5, var_e=var_e, reps=rep_yt5, sim_param=SP)
        FemaleHybridYT5 = set_pheno(FemaleHybridYT5, var_e=var_e, reps=rep_yt5, sim_param=SP)
        
        MaleInbredYT5_ids = list(set(MaleHybridYT5.mother))
        FemaleInbredYT5_ids = list(set(FemaleHybridYT5.mother))
        MaleInbredYT5 = select_ind(MaleInbredYT4, n_ind=len(MaleInbredYT5_ids),
                                parents=[MaleInbredYT4.id.index(id) for id in MaleInbredYT5_ids if id in MaleInbredYT4.id], sim_param=SP)
        FemaleInbredYT5 = select_ind(FemaleInbredYT4, n_ind=len(FemaleInbredYT5_ids),
                                   parents=[FemaleInbredYT4.id.index(id) for id in FemaleInbredYT5_ids if id in FemaleInbredYT4.id], sim_param=SP)
        
        # Stage 5
        MaleHybridYT4 = select_ind(MaleHybridYT3, n_ind=n_yt4, use="pheno", sim_param=SP)
        FemaleHybridYT4 = select_ind(FemaleHybridYT3, n_ind=n_yt4, use="pheno", sim_param=SP)
        
        MaleHybridYT4 = set_pheno(MaleHybridYT4, var_e=var_e, reps=rep_yt4, sim_param=SP)
        FemaleHybridYT4 = set_pheno(FemaleHybridYT4, var_e=var_e, reps=rep_yt4, sim_param=SP)
        
        MaleInbredYT4_ids = list(set(MaleHybridYT4.mother))
        FemaleInbredYT4_ids = list(set(FemaleHybridYT4.mother))
        MaleInbredYT4 = select_ind(MaleInbredYT3, n_ind=len(MaleInbredYT4_ids),
                                 parents=[MaleInbredYT3.id.index(id) for id in MaleInbredYT4_ids if id in MaleInbredYT3.id], sim_param=SP)
        FemaleInbredYT4 = select_ind(FemaleInbredYT3, n_ind=len(FemaleInbredYT4_ids),
                                   parents=[FemaleInbredYT3.id.index(id) for id in FemaleInbredYT4_ids if id in FemaleInbredYT3.id], sim_param=SP)
        
        # Stage 4
        MaleInbredYT3 = select_ind(MaleYT2, n_ind=n_inbred3, use="pheno", sim_param=SP)
        FemaleInbredYT3 = select_ind(FemaleYT2, n_ind=n_inbred3, use="pheno", sim_param=SP)
        
        MaleHybridYT3 = hybrid_cross(MaleInbredYT3, FemaleElite, return_hybrid_pop=True, sim_param=SP)
        FemaleHybridYT3 = hybrid_cross(FemaleInbredYT3, MaleElite, return_hybrid_pop=True, sim_param=SP)
        
        MaleHybridYT3 = set_pheno(MaleHybridYT3, var_e=var_e, reps=rep_yt3, sim_param=SP)
        FemaleHybridYT3 = set_pheno(FemaleHybridYT3, var_e=var_e, reps=rep_yt3, sim_param=SP)
        
        # Stage 3
        # Report selection accuracy
        if MaleYT1.n_ind > 0 and FemaleYT1.n_ind > 0:
            male_cor = np.corrcoef(MaleYT1.pheno[:, 0], MaleYT1.gv[:, 0])[0, 1]
            female_cor = np.corrcoef(FemaleYT1.pheno[:, 0], FemaleYT1.gv[:, 0])[0, 1]
            output['acc_sel'][year - 1] = (male_cor + female_cor) / 2
        
        MaleYT2 = select_ind(MaleYT1, n_ind=n_inbred2, use="pheno", sim_param=SP)
        FemaleYT2 = select_ind(FemaleYT1, n_ind=n_inbred2, use="pheno", sim_param=SP)
        
        MaleYT2 = set_pheno_gca(MaleYT2, FemaleTester2, reps=rep_yt2, inbred=True, sim_param=SP)
        FemaleYT2 = set_pheno_gca(FemaleYT2, MaleTester2, reps=rep_yt2, inbred=True, sim_param=SP)
        
        # Stage 2
        MaleDH = make_dh(MaleF1, n_dh=n_dh, sim_param=SP)
        FemaleDH = make_dh(FemaleF1, n_dh=n_dh, sim_param=SP)
        
        MaleYT1 = set_pheno_gca(MaleDH, FemaleTester1, reps=rep_yt1, inbred=True, sim_param=SP)
        FemaleYT1 = set_pheno_gca(FemaleDH, MaleTester1, reps=rep_yt1, inbred=True, sim_param=SP)
        
        # Stage 1
        MaleF1 = rand_cross(MaleParents, n_crosses=n_crosses, sim_param=SP)
        FemaleF1 = rand_cross(FemaleParents, n_crosses=n_crosses, sim_param=SP)
        
        # Report results
        output['mean_g_inbred'][year - 1] = (mean_g(MaleInbredYT3)[0] + mean_g(FemaleInbredYT3)[0]) / 2
        output['var_g_inbred'][year - 1] = (var_g(MaleInbredYT3)[0] + var_g(FemaleInbredYT3)[0]) / 2
        
        tmp_hybrid = hybrid_cross(FemaleInbredYT3, MaleInbredYT3, return_hybrid_pop=True, sim_param=SP)
        output['mean_g_hybrid'][year - 1] = mean_g(tmp_hybrid)[0]
        output['var_g_hybrid'][year - 1] = var_g(tmp_hybrid)[0]
        
        tmp_gca = calc_gca(tmp_hybrid, use="gv")
        inbred_gv = np.concatenate([FemaleInbredYT3.gv[:, 0], MaleInbredYT3.gv[:, 0]])
        gca_values = np.concatenate([tmp_gca['GCAf'][:, 1], tmp_gca['GCAm'][:, 1]])
        output['cor'][year - 1] = np.corrcoef(inbred_gv, gca_values)[0, 1]
    
    # Future phase: Phenotypic program
    print("--> Working on Phenotypic hybrid program")
    for year in range(n_burnin + 1, n_burnin + n_future + 1):
        print(f" Working on future year: {year}")
        
        # Update parents (pick new parents)
        MaleParents_new = select_ind(MaleInbredYT4, n_ind=10, use="pheno", sim_param=SP)
        MaleParents_old = select_ind(MaleParents, n_ind=n_parents - 10,
                                  parents=list(range(10, n_parents)), sim_param=SP)
        MaleParents = merge_pops([MaleParents_old, MaleParents_new])
        
        FemaleParents_new = select_ind(FemaleInbredYT4, n_ind=10, use="pheno", sim_param=SP)
        FemaleParents_old = select_ind(FemaleParents, n_ind=n_parents - 10,
                                    parents=list(range(10, n_parents)), sim_param=SP)
        FemaleParents = merge_pops([FemaleParents_old, FemaleParents_new])
        
        # Update testers (pick new testers)
        best_male_idx = np.argmax(MaleHybridYT5.pheno[:, 0])
        best_male_inbred_id = MaleHybridYT5.mother[best_male_idx]
        best_male_inbred_idx = MaleInbredYT5.id.index(best_male_inbred_id)
        MaleElite_new = select_ind(MaleInbredYT5, n_ind=1, parents=[best_male_inbred_idx], sim_param=SP)
        MaleElite_old = select_ind(MaleElite, n_ind=n_elite - 1, parents=list(range(1, n_elite)), sim_param=SP)
        MaleElite = merge_pops([MaleElite_old, MaleElite_new])
        
        best_female_idx = np.argmax(FemaleHybridYT5.pheno[:, 0])
        best_female_inbred_id = FemaleHybridYT5.mother[best_female_idx]
        best_female_inbred_idx = FemaleInbredYT5.id.index(best_female_inbred_id)
        FemaleElite_new = select_ind(FemaleInbredYT5, n_ind=1, parents=[best_female_inbred_idx], sim_param=SP)
        FemaleElite_old = select_ind(FemaleElite, n_ind=n_elite - 1, parents=list(range(1, n_elite)), sim_param=SP)
        FemaleElite = merge_pops([FemaleElite_old, FemaleElite_new])
        
        MaleTester1 = select_ind(MaleElite, n_ind=n_tester1, parents=list(range(n_tester1)), sim_param=SP)
        FemaleTester1 = select_ind(FemaleElite, n_ind=n_tester1, parents=list(range(n_tester1)), sim_param=SP)
        MaleTester2 = select_ind(MaleElite, n_ind=n_tester2, parents=list(range(n_tester2)), sim_param=SP)
        FemaleTester2 = select_ind(FemaleElite, n_ind=n_tester2, parents=list(range(n_tester2)), sim_param=SP)
        
        # Advance year (advances yield trials by a year)
        p = P[year - 1]
        
        # Stage 6
        MaleHybridYT5 = select_ind(MaleHybridYT4, n_ind=n_yt5, use="pheno", sim_param=SP)
        FemaleHybridYT5 = select_ind(FemaleHybridYT4, n_ind=n_yt5, use="pheno", sim_param=SP)
        
        MaleHybridYT5 = set_pheno(MaleHybridYT5, var_e=var_e, reps=rep_yt5, sim_param=SP)
        FemaleHybridYT5 = set_pheno(FemaleHybridYT5, var_e=var_e, reps=rep_yt5, sim_param=SP)
        
        MaleInbredYT5_ids = list(set(MaleHybridYT5.mother))
        FemaleInbredYT5_ids = list(set(FemaleHybridYT5.mother))
        MaleInbredYT5 = select_ind(MaleInbredYT4, n_ind=len(MaleInbredYT5_ids),
                                parents=[MaleInbredYT4.id.index(id) for id in MaleInbredYT5_ids if id in MaleInbredYT4.id], sim_param=SP)
        FemaleInbredYT5 = select_ind(FemaleInbredYT4, n_ind=len(FemaleInbredYT5_ids),
                                   parents=[FemaleInbredYT4.id.index(id) for id in FemaleInbredYT5_ids if id in FemaleInbredYT4.id], sim_param=SP)
        
        # Stage 5
        MaleHybridYT4 = select_ind(MaleHybridYT3, n_ind=n_yt4, use="pheno", sim_param=SP)
        FemaleHybridYT4 = select_ind(FemaleHybridYT3, n_ind=n_yt4, use="pheno", sim_param=SP)
        
        MaleHybridYT4 = set_pheno(MaleHybridYT4, var_e=var_e, reps=rep_yt4, sim_param=SP)
        FemaleHybridYT4 = set_pheno(FemaleHybridYT4, var_e=var_e, reps=rep_yt4, sim_param=SP)
        
        MaleInbredYT4_ids = list(set(MaleHybridYT4.mother))
        FemaleInbredYT4_ids = list(set(FemaleHybridYT4.mother))
        MaleInbredYT4 = select_ind(MaleInbredYT3, n_ind=len(MaleInbredYT4_ids),
                                 parents=[MaleInbredYT3.id.index(id) for id in MaleInbredYT4_ids if id in MaleInbredYT3.id], sim_param=SP)
        FemaleInbredYT4 = select_ind(FemaleInbredYT3, n_ind=len(FemaleInbredYT4_ids),
                                   parents=[FemaleInbredYT3.id.index(id) for id in FemaleInbredYT4_ids if id in FemaleInbredYT3.id], sim_param=SP)
        
        # Stage 4
        MaleInbredYT3 = select_ind(MaleYT2, n_ind=n_inbred3, use="pheno", sim_param=SP)
        FemaleInbredYT3 = select_ind(FemaleYT2, n_ind=n_inbred3, use="pheno", sim_param=SP)
        
        MaleHybridYT3 = hybrid_cross(MaleInbredYT3, FemaleElite, return_hybrid_pop=True, sim_param=SP)
        FemaleHybridYT3 = hybrid_cross(FemaleInbredYT3, MaleElite, return_hybrid_pop=True, sim_param=SP)
        
        MaleHybridYT3 = set_pheno(MaleHybridYT3, var_e=var_e, reps=rep_yt3, sim_param=SP)
        FemaleHybridYT3 = set_pheno(FemaleHybridYT3, var_e=var_e, reps=rep_yt3, sim_param=SP)
        
        # Stage 3
        MaleYT2 = select_ind(MaleYT1, n_ind=n_inbred2, use="pheno", sim_param=SP)
        FemaleYT2 = select_ind(FemaleYT1, n_ind=n_inbred2, use="pheno", sim_param=SP)
        
        MaleYT2 = set_pheno_gca(MaleYT2, FemaleTester2, reps=rep_yt2, inbred=True, sim_param=SP)
        FemaleYT2 = set_pheno_gca(FemaleYT2, MaleTester2, reps=rep_yt2, inbred=True, sim_param=SP)
        
        # Stage 2
        MaleDH = make_dh(MaleF1, n_dh=n_dh, sim_param=SP)
        FemaleDH = make_dh(FemaleF1, n_dh=n_dh, sim_param=SP)
        
        MaleYT1 = set_pheno_gca(MaleDH, FemaleTester1, reps=rep_yt1, inbred=True, sim_param=SP)
        FemaleYT1 = set_pheno_gca(FemaleDH, MaleTester1, reps=rep_yt1, inbred=True, sim_param=SP)
        
        # Stage 1
        MaleF1 = rand_cross(MaleParents, n_crosses=n_crosses, sim_param=SP)
        FemaleF1 = rand_cross(FemaleParents, n_crosses=n_crosses, sim_param=SP)
        
        # Report results
        output['mean_g_inbred'][year - 1] = (mean_g(MaleInbredYT3)[0] + mean_g(FemaleInbredYT3)[0]) / 2
        output['var_g_inbred'][year - 1] = (var_g(MaleInbredYT3)[0] + var_g(FemaleInbredYT3)[0]) / 2
        
        tmp_hybrid = hybrid_cross(FemaleInbredYT3, MaleInbredYT3, return_hybrid_pop=True, sim_param=SP)
        output['mean_g_hybrid'][year - 1] = mean_g(tmp_hybrid)[0]
        output['var_g_hybrid'][year - 1] = var_g(tmp_hybrid)[0]
        
        tmp_gca = calc_gca(tmp_hybrid, use="gv")
        inbred_gv = np.concatenate([FemaleInbredYT3.gv[:, 0], MaleInbredYT3.gv[:, 0]])
        gca_values = np.concatenate([tmp_gca['GCAf'][:, 1], tmp_gca['GCAm'][:, 1]])
        output['cor'][year - 1] = np.corrcoef(inbred_gv, gca_values)[0, 1]
    
    # Save results from current replicate
    results.append(output)

print("\nSimulation completed!")

## Analyze Results

Visualize the results from the simulation.

In [ ]:
# Combine results from all replicates
df = pd.DataFrame(results[0])  # For single replicate, convert dict to DataFrame

# If multiple replicates, combine them
if len(results) > 1:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

print("Results summary:")
print(df.head(10))
print(f"\nTotal years simulated: {len(df)}")

In [ ]:
# Plotting function
def plot_results(x, y, title, xlabel, ylabel, ylim=None):
    plt.plot(x, y, 'b-', linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(ylim)
    plt.grid(True, linestyle='--', alpha=0.7)

# Create plots
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# Inbred Genetic Gain
plt.sca(axes[0, 0])
plot_results(df['year'], df['mean_g_inbred'], 
              'Inbred genetic gain', 'Year', 'Yield')

# Hybrid Genetic Gain
plt.sca(axes[0, 1])
plot_results(df['year'], df['mean_g_hybrid'], 
              'Hybrid genetic gain', 'Year', 'Yield')

# Inbred Variance
plt.sca(axes[1, 0])
plot_results(df['year'], df['var_g_inbred'], 
              'Inbred genetic variance', 'Year', 'Variance')

# Hybrid Variance
plt.sca(axes[1, 1])
plot_results(df['year'], df['var_g_hybrid'], 
              'Hybrid genetic variance', 'Year', 'Variance')

# Selection Accuracy
plt.sca(axes[2, 0])
plot_results(df['year'], df['acc_sel'], 
              'Selection accuracy', 'Year', 'Accuracy')

# Correlation
plt.sca(axes[2, 1])
plot_results(df['year'], df['cor'], 
              'Inbred vs. hybrid yield cor.', 'Year', 'Correlation')

plt.tight_layout()
plt.savefig('PhenotypicSelection_Results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Results plot saved as 'PhenotypicSelection_Results.png'")

## Summary

This tutorial demonstrated:

1. **Founder Population Creation**: Using `runMacs` with heterotic pool split to generate initial haplotypes
2. **Trait Definition**: Adding traits with additive, dominance, and GxE effects using `addTraitADG`
3. **Hybrid Breeding Pipeline**: Setting up a 6-stage hybrid breeding pipeline with:
   - F1 crosses
   - Doubled haploid (DH) line production
   - GCA evaluation using testers (YT1, YT2)
   - Hybrid evaluation (YT3, YT4, YT5)
4. **Phenotypic Selection**: Selecting superior inbreds and hybrids at each stage based on phenotypic performance
5. **Parent and Tester Updates**: Replacing parents and testers based on performance
6. **Genetic Progress**: Tracking inbred and hybrid genetic gain, variance, selection accuracy, and correlations over time

The simulation shows how phenotypic selection can be used in hybrid breeding programs to improve genetic gain over multiple generations while maintaining separate heterotic pools.